In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# ============================================================
# 1. LOAD ALL MODEL RESULTS
# ============================================================
models = {}
for name in ['M0', 'M1', 'M2']:
    df = pd.read_csv(f'../output/results/{name}_results.csv')
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['country', 'date']).reset_index(drop=True)
    models[name] = df

# ============================================================
# 2. DEFINE GROUPS
# ============================================================
group_map = {
    'Saudi Arabia': 'Oil Exporter',
    'Abu Dhabi': 'Oil Exporter',
    'Dubai': 'Oil Exporter',
    'Qatar': 'Oil Exporter',
    'Colombia': 'Oil Exporter',
    'Mexico': 'Oil Exporter',
    'Brazil': 'Oil Exporter',
    'Egypt': 'Oil Exporter',
    'Malaysia': 'Oil Exporter',
    'Indonesia': 'Control',
    'Philippines': 'Control',
    'Turkey': 'Control',
    'Chile': 'Control',
    'China': 'Control',
    'South Africa': 'Control',
    'South Korea': 'Control',
    'Thailand': 'Control',
}

# ============================================================
# 3. NEWEY-WEST CORRECTED CORRELATION
# ============================================================
def nw_pvalue(x, y, nlags=None):
    """Correlation t-test with Newey-West HAC standard errors."""
    n = len(x)
    if nlags is None:
        nlags = int(n ** (1/3))
    xd = x - x.mean()
    yd = y - y.mean()
    xy = xd * yd
    gamma0 = np.var(xy, ddof=1)
    nw = gamma0
    for j in range(1, nlags + 1):
        w = 1 - j / (nlags + 1)
        nw += 2 * w * np.cov(xy[j:], xy[:-j], ddof=1)[0, 1]
    t = np.mean(xy) / np.sqrt(nw / n)
    return 2 * (1 - stats.t.cdf(abs(t), df=n - 2))

def star(p):
    if pd.isna(p): return ''
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

# ============================================================
# 4. COMPUTE CORRELATIONS PER MODEL x COUNTRY x HORIZON
# ============================================================
horizons = [1, 4, 12]
horizon_labels = ['1w', '1m', '3m']

def compute_correlations(df, dd_col='distance_to_distress', cds_col='cds_spread'):
    rows = []
    for country in sorted(df['country'].unique()):
        d = df[df['country'] == country].sort_values('date').copy()
        group = group_map.get(country, 'Control')
        row = {'Country': country, 'Group': group}

        for h, hlbl in zip(horizons, horizon_labels):
            dx = d[dd_col].diff(h)
            dy = d[cds_col].diff(h)
            valid = dx.notna() & dy.notna()
            x, y = dx[valid].values, dy[valid].values

            if len(x) < 20:
                for k in ['rho', 'p', 'rhoS', 'Dir']:
                    row[f'{k}_{hlbl}'] = np.nan
                continue

            rho, _ = stats.pearsonr(x, y)
            p_nw = nw_pvalue(x, y)
            rho_s, _ = stats.spearmanr(x, y)
            nz = (x != 0) & (y != 0)
            dir_acc = np.mean((x[nz] > 0) == (y[nz] < 0)) if nz.sum() > 0 else np.nan

            row[f'rho_{hlbl}'] = rho
            row[f'p_{hlbl}'] = p_nw
            row[f'rhoS_{hlbl}'] = rho_s
            row[f'Dir_{hlbl}'] = dir_acc

        rows.append(row)
    return pd.DataFrame(rows)

corr_by_model = {}
for name, df in models.items():
    corr_by_model[name] = compute_correlations(df)

# ============================================================
# 5. COMPARISON TABLE PER HORIZON
# ============================================================
def build_comparison(corr_dict, hlbl):
    rows = []
    ref = corr_dict['M0']

    for _, r0 in ref.iterrows():
        country = r0['Country']
        group = r0['Group']
        row = {'Country': country, 'Group': group}

        for mname in ['M0', 'M1', 'M2']:
            rc = corr_dict[mname]
            r = rc[rc['Country'] == country].iloc[0]
            rho = r[f'rho_{hlbl}']
            p = r[f'p_{hlbl}']
            rho_s = r[f'rhoS_{hlbl}']
            d = r[f'Dir_{hlbl}']

            row[f'{mname}_rho'] = f"{rho:.2f}{star(p)}" if pd.notna(rho) else ''
            row[f'{mname}_rhoS'] = f"{rho_s:.2f}" if pd.notna(rho_s) else ''
            row[f'{mname}_Dir'] = f"{d:.0%}" if pd.notna(d) else ''

        rows.append(row)

    tbl = pd.DataFrame(rows)

    # Group averages
    for gname in ['Oil Exporter', 'Control']:
        avg = {'Country': f'--- {gname} avg ---', 'Group': gname}
        for mname in ['M0', 'M1', 'M2']:
            raw = corr_dict[mname]
            g = raw[raw['Group'] == gname]
            avg[f'{mname}_rho'] = f"{g[f'rho_{hlbl}'].dropna().mean():.3f}"
            avg[f'{mname}_rhoS'] = f"{g[f'rhoS_{hlbl}'].dropna().mean():.3f}"
            avg[f'{mname}_Dir'] = f"{g[f'Dir_{hlbl}'].dropna().mean():.0%}"
        rows.append(avg)

    # DiD
    did = {'Country': '--- DiD (Exp-Ctrl) ---', 'Group': ''}
    for mname in ['M0', 'M1', 'M2']:
        raw = corr_dict[mname]
        e = raw[raw['Group'] == 'Oil Exporter'][f'rho_{hlbl}'].dropna().mean()
        c = raw[raw['Group'] == 'Control'][f'rho_{hlbl}'].dropna().mean()
        eS = raw[raw['Group'] == 'Oil Exporter'][f'rhoS_{hlbl}'].dropna().mean()
        cS = raw[raw['Group'] == 'Control'][f'rhoS_{hlbl}'].dropna().mean()
        did[f'{mname}_rho'] = f"{e - c:+.3f}"
        did[f'{mname}_rhoS'] = f"{eS - cS:+.3f}"
        did[f'{mname}_Dir'] = ''
    rows.append(did)

    return pd.DataFrame(rows)

# ============================================================
# 6. PRINT
# ============================================================
for hlbl in horizon_labels:
    print(f"\n{'='*120}")
    print(f"  HORIZON: {hlbl}  |  Dd2 vs DCDS  |  Newey-West p-values")
    print(f"{'='*120}")
    tbl = build_comparison(corr_by_model, hlbl)
    # Sort: exporters first
    exp = tbl[tbl['Group'] == 'Oil Exporter']
    ctrl = tbl[tbl['Group'] == 'Control']
    summ = tbl[tbl['Group'].isin(['', 'Oil Exporter', 'Control']) & tbl['Country'].str.startswith('---')]
    body = pd.concat([exp[~exp['Country'].str.startswith('---')],
                      ctrl[~ctrl['Country'].str.startswith('---')],
                      summ])
    print(body.to_string(index=False))

# ============================================================
# 7. COMPACT SUMMARY
# ============================================================
print(f"\n{'='*80}")
print("  Mean Pearson rho by Group x Model x Horizon")
print(f"{'='*80}")
rows = []
for hlbl in horizon_labels:
    for gname in ['Oil Exporter', 'Control']:
        row = {'Horizon': hlbl, 'Group': gname}
        for mname in ['M0', 'M1', 'M2']:
            raw = corr_by_model[mname]
            row[mname] = round(raw[raw['Group'] == gname][f'rho_{hlbl}'].dropna().mean(), 4)
        rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print(f"\n  DiD = Exporter avg - Control avg")
rows = []
for hlbl in horizon_labels:
    row = {'Horizon': hlbl}
    for mname in ['M0', 'M1', 'M2']:
        raw = corr_by_model[mname]
        e = raw[raw['Group'] == 'Oil Exporter'][f'rho_{hlbl}'].dropna().mean()
        c = raw[raw['Group'] == 'Control'][f'rho_{hlbl}'].dropna().mean()
        row[mname] = round(e - c, 4)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))


  HORIZON: 1w  |  Dd2 vs DCDS  |  Newey-West p-values
                 Country        Group   M0_rho M0_rhoS M0_Dir   M1_rho M1_rhoS M1_Dir   M2_rho M2_rhoS M2_Dir
               Abu Dhabi Oil Exporter     0.01    0.04    48%    -0.07   -0.04    56%  -0.12**   -0.07    51%
                  Brazil Oil Exporter -0.21***   -0.28    61% -0.28***   -0.31    63% -0.15***   -0.33    63%
                Colombia Oil Exporter -0.12***   -0.31    64%  -0.15**   -0.31    64% -0.13***   -0.35    64%
                   Dubai Oil Exporter    -0.02   -0.00    48%    -0.15   -0.06    56% -0.12***   -0.12    53%
                   Egypt Oil Exporter     0.00   -0.09    51%    -0.00   -0.08    49%    -0.04   -0.09    52%
                Malaysia Oil Exporter    -0.02   -0.21    62%    -0.08   -0.18    58% -0.08***   -0.27    64%
                  Mexico Oil Exporter    -0.28   -0.18    59%    -0.33   -0.22    60% -0.14***   -0.25    61%
                   Qatar Oil Exporter     0.00    0.07    47%    